# Electricity NPV summary

Run the electricity Monte Carlo NPV distribution, deterministic NPV, and Monte Carlo ranking summaries. The distribution table includes the number and share of simulations with non-negative NPV (NPV >= 0) and the number with negative NPV. This notebook displays figures inline only; it does not save figures or CSV files.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from electricity.electricity_npv_deterministic import calculate_deterministic_electricity_results
from electricity.electricity_npv_monte_carlo import DEFAULT_RANDOM_SEED, DEFAULT_SAMPLE_SIZE, simulate_electricity_results
from electricity.electricity_npv_summary_figures import (
    ELECTRICITY_FINANCIAL_METRIC_OPTIONS,
    ELECTRICITY_TECHNOLOGY_LABELS,
    calculate_electricity_npv_rankings_from_results,
)
from npv_summary import summarize_metric_signs
from npv_summary_plots import plot_average_rank_bars, plot_financial_metric_technology_bars


## Settings

In [2]:
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED
TECHNOLOGIES = tuple(ELECTRICITY_TECHNOLOGY_LABELS)
FINANCIAL_METRIC = "LNM"  # "NPV", "LNM", or "LCOX"

if FINANCIAL_METRIC not in ELECTRICITY_FINANCIAL_METRIC_OPTIONS:
    raise ValueError(f"Unknown FINANCIAL_METRIC: {FINANCIAL_METRIC!r}")

METRIC_CONFIG = ELECTRICITY_FINANCIAL_METRIC_OPTIONS[FINANCIAL_METRIC]
METRIC_COLUMN = str(METRIC_CONFIG["metric_column"])
METRIC_SCALE_FACTOR = float(METRIC_CONFIG["scale"])
METRIC_SUMMARY_COLUMN = str(METRIC_CONFIG["summary_column"])
METRIC_AXIS_LABEL = str(METRIC_CONFIG["axis_label"])
METRIC_RANKING_LABEL = str(METRIC_CONFIG["ranking_label"])
METRIC_HIGHER_IS_BETTER = bool(METRIC_CONFIG["higher_is_better"])
METRIC_COLOR_BY_SIGN = bool(METRIC_CONFIG["color_by_sign"])
METRIC_ZERO_BASELINE = bool(METRIC_CONFIG["zero_baseline"])

pd.options.display.float_format = "{:,.3f}".format


## Monte Carlo NPV distribution

In [3]:
monte_carlo_results = simulate_electricity_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=TECHNOLOGIES,
)

mc_summary_rows = []
for technology, results in monte_carlo_results.items():
    values = np.asarray(results[METRIC_COLUMN], dtype=float) / METRIC_SCALE_FACTOR
    mc_summary_rows.append(
        {
            "label": ELECTRICITY_TECHNOLOGY_LABELS.get(technology, technology),
            METRIC_SUMMARY_COLUMN: values.mean(),
            "median": np.median(values),
            "p05": np.percentile(values, 5),
            "p95": np.percentile(values, 95),
            **summarize_metric_signs(values),
        }
    )
mc_summary = pd.DataFrame(mc_summary_rows).sort_values(
    METRIC_SUMMARY_COLUMN, ascending=not METRIC_HIGHER_IS_BETTER
)
mc_summary_by_label = mc_summary.set_index("label")

display(mc_summary)
plot_financial_metric_technology_bars(
    values=mc_summary_by_label[METRIC_SUMMARY_COLUMN].to_dict(),
    output_path=None,
    title=f"Monte Carlo mean {METRIC_RANKING_LABEL} by electricity technology",
    median_values=mc_summary_by_label["median"].to_dict(),
    lower_values=mc_summary_by_label["p05"].to_dict(),
    upper_values=mc_summary_by_label["p95"].to_dict(),
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    x_axis_label=METRIC_AXIS_LABEL,
    higher_is_better=METRIC_HIGHER_IS_BETTER,
    color_by_sign=METRIC_COLOR_BY_SIGN,
    zero_baseline=METRIC_ZERO_BASELINE,
)
plt.show()


,label,levelized_net_margin_eur_per_mwh,median,p05,p95,non_negative_count,negative_count,non_negative_share
7,PV,17.490,17.491,10.002,25.039,100000,0,1.000
6,Wind Onshore,13.680,13.672,3.305,24.089,99859,141,0.999
5,Wind Offshore,7.253,7.295,-6.026,20.472,74896,25104,0.749
2,CCGT,-28.887,-26.408,-71.863,5.533,10627,89373,0.106
3,CCGT + CCS,-37.677,-34.962,-87.993,3.194,7254,92746,0.073
4,Nuclear,-49.091,-48.934,-96.264,-2.096,2953,97047,0.030
0,Hard coal,-64.690,-64.163,-77.497,-53.653,0,100000,0.000
1,Hard coal + CCS,-69.697,-69.735,-94.936,-44.740,0,100000,0.000
8,Biogas,-245.056,-245.045,-277.296,-212.651,0,100000,0.000


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42865/3433122035.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Deterministic NPV

In [4]:
deterministic_results = calculate_deterministic_electricity_results(technologies=TECHNOLOGIES)
det_summary_rows = []
for technology, results in deterministic_results.items():
    det_summary_rows.append(
        {
            "label": ELECTRICITY_TECHNOLOGY_LABELS.get(technology, technology),
            METRIC_SUMMARY_COLUMN: float(np.asarray(results[METRIC_COLUMN]).item()) / METRIC_SCALE_FACTOR,
        }
    )
det_summary = pd.DataFrame(det_summary_rows).sort_values(
    METRIC_SUMMARY_COLUMN, ascending=not METRIC_HIGHER_IS_BETTER
)

display(det_summary)
plot_financial_metric_technology_bars(
    values=det_summary.set_index("label")[METRIC_SUMMARY_COLUMN].to_dict(),
    output_path=None,
    title=f"Deterministic {METRIC_RANKING_LABEL} by electricity technology",
    x_axis_label=METRIC_AXIS_LABEL,
    higher_is_better=METRIC_HIGHER_IS_BETTER,
    color_by_sign=METRIC_COLOR_BY_SIGN,
    zero_baseline=METRIC_ZERO_BASELINE,
)
plt.show()


,label,levelized_net_margin_eur_per_mwh
7,PV,17.897
6,Wind Onshore,14.316
5,Wind Offshore,7.810
2,CCGT,-28.442
3,CCGT + CCS,-37.566
4,Nuclear,-48.563
0,Hard coal,-63.886
1,Hard coal + CCS,-68.084
8,Biogas,-242.540


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42865/3049702141.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Monte Carlo NPV ranking

In [5]:
ranking_raw, ranking_summary = calculate_electricity_npv_rankings_from_results(
    results=monte_carlo_results,
    sector_name="Electricity",
    financial_metric=FINANCIAL_METRIC,
)

ranking_summary_for_plot = ranking_summary.assign(
    display_label=ranking_summary["technology"].map(ELECTRICITY_TECHNOLOGY_LABELS).fillna(ranking_summary["technology"])
)
rank_table = (
    ranking_summary_for_plot.rename(columns={"display_label": "Technology"})
    .loc[:, ["Technology", "average_rank", "probability_rank_1", "probability_top_3", "n_simulations"]]
    .rename(
        columns={
            "average_rank": "Average rank",
            "probability_rank_1": "Probability rank 1",
            "probability_top_3": "Probability top 3",
            "n_simulations": "Simulations",
        }
    )
)
display(rank_table)
plot_average_rank_bars(
    ranking_summary=ranking_summary_for_plot,
    output_path=None,
    title=f"Monte Carlo {METRIC_RANKING_LABEL} Ranking",
    metric_label=METRIC_RANKING_LABEL,
    random_seed=RANDOM_SEED,
    higher_is_better=METRIC_HIGHER_IS_BETTER,
)
plt.show()


,Technology,Average rank,Probability rank 1,Probability top 3,Simulations
0,PV,1.513,0.579,0.998,100000
1,Wind Onshore,1.987,0.303,0.983,100000
2,Wind Offshore,2.664,0.115,0.919,100000
3,CCGT,4.534,0.001,0.052,100000
4,CCGT + CCS,5.546,0.002,0.031,100000
5,Nuclear,5.932,0.000,0.017,100000
6,Hard coal,6.764,0.000,0.000,100000
7,Hard coal + CCS,7.058,0.000,0.000,100000
8,Biogas,9.000,0.000,0.000,100000


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42865/2781369593.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
